<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Manipulating and Visualising Data
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

## How to use this notebook

Most of this notebook is worked examples: usually I demo them, and you read and run them as we go. Two kinds of activity are marked with **»**:

* **» Predict** — *before* running, write down/discuss what you think the output will be, then run to check.
* **» Your Turn** — you type the code. There is always a worked example just above to copy the pattern from. Blanks look like `____`: delete the whole `____` and type your own code in its place, e.g. `____ = 128` becomes `systolic = 128`.

This follows the PRIMM pattern used in programming teaching: Predict, Run, Investigate, Modify, Make.

Part way through, an **Independent practice** slide hands over to you for a stretch of self-paced work; a **Regroup** slide marks where we come back together.

The notebook is **one environment**: an import or a variable made in one cell exists in every cell after it, so the cells build on each other and are meant to be run in order. After a restart, or if a cell complains that `pd` or a `df_...` is not defined, use **Runtime → Run before** to catch up. The capstone and the Additional work stand on their own; everything else builds on the cells before it.

> Worked solutions, with explanations, are in the solutions notebook, released after the session.

## Welcome

**Goal:** by the end of today you will load a clinical table, ask it a question, and plot the answer.

Last session was Python itself. Today is the two libraries you will use in every later session: **pandas** for tables and **matplotlib** for plots. As before, most of the session is hands-on and in pairs, with a long stretch of independent practice in the middle, and a capstone to take home. If you are stuck, say so.

**Your prep presentations** come after the first section, at the start of the tables material, where they belong.

### Before we start
1. Open this notebook in Colab, from the **Google Colab** link on the course material, or directly: [Open in Colab](https://colab.research.google.com/github/stuartbowyer/sdsai-lecture-notes/blob/main/notebooks/02_Manipulating_and_Visualising_Data.ipynb).
2. **File → Save a copy in Drive**, and work in your copy.
3. Check the runtime is *Connected* (top right).

## Intended Learning Outcomes
By the end of this session you will be able to:

1. **Write** reusable functions with parameters, default values and documentation, and **import** functionality from Python modules and libraries.
2. **Load** tabular data into Python and **manipulate** it to answer a question about it.
3. **Convert** text-encoded measurements into numeric values, and **summarise** a table with descriptive statistics, overall and by group.
4. **Combine** related tables, and **explain** how the choice of method determines which records are kept.
5. **Produce** a plot that answers a clinical question, and **justify** the choice of plot type.

## Session outline
1. Functions and imports
2. CRP results for one patient, with lists
3. Tables in pandas: your feature recipes, then load, inspect, select, filter
4. Independent practice: the rest of the table basics, then files, text columns, statistics, grouping
5. Combining tables
6. Plotting with matplotlib and pandas
7. Closing
8. At home: the capstone, then additional work and appendices

# 1. Functions
* Almost everything you did last session used functions: `print()`, `len()`, `type()`, `sorted()`.
* A function takes **inputs** (its parameters) and may **return** an output.
* Functions attached to an object are called **methods**, and are called with a dot: `procedures.sort()`.
* Writing your own lets you reuse code instead of copying it, and name what it does.

In [ ]:
procedures = ['Hernia repair', 'Appendicectomy', 'Cholecystectomy', 'Colectomy', 'Mastectomy']

# len is a function with:
#   - a sequence (the list) as an input
#   - an integer as an output
print(len(procedures))

# reverse is a method of the list object, with:
#   - no inputs
#   - no outputs (it modifies the list in place)
procedures.reverse()
print(procedures)

## Creating a function
* `def` starts the definition; the parameters go in the brackets; `return` sends the result back to whoever called the function.
* The indented lines are the **body**, as with `if` and `for` last session.
* Here we calculate BMI from height and weight.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    bmi = weight_kg / height_m ** 2
    return bmi

# Call the function; the notebook shows what it returns
calculate_bmi(1.80, 90)

**» Predict** — The same function, called with the two arguments the other way round. **Is the result an error, a wrong number, or the right number?** Write it down, then run.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    bmi = weight_kg / height_m ** 2
    return bmi

calculate_bmi(90, 1.80)

### Positional and keyword arguments
* Python matched the arguments **by position**: `90` became `height_m`, so the call ran happily and gave a nonsense BMI. **A wrong answer is worse than an error**, because nothing tells you.
* Naming the arguments (**keyword arguments**) makes the order irrelevant and the call readable:

In [ ]:
def calculate_bmi(height_m, weight_kg):
    bmi = weight_kg / height_m ** 2
    return bmi

calculate_bmi(weight_kg=90, height_m=1.80)

**» Predict** — `bmi` is created inside the function. After the call, **does the last line print the BMI, or give an error?** Decide, then run.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    bmi = weight_kg / height_m ** 2
    return bmi

result = calculate_bmi(1.80, 90)
print(result)
print(bmi)

### Variables inside a function are local
* `bmi` exists only while the function runs. Outside it Python has never heard of it, hence the `NameError`.
* The **only** thing that leaves a function is what you `return`. Store it in a variable (`result`) if you need it afterwards.

### Default parameters
* A parameter can have a **default value**, used when the caller leaves it out.
* The default is the common case; the caller overrides it for the exception.
* Here a CRP result is flagged against the usual cut-off of 5 mg/L, unless a different threshold is given, for example in the first days after surgery when CRP is raised anyway.

In [ ]:
def is_elevated(crp_mg_l, threshold=5.0):
    return crp_mg_l > threshold

print(is_elevated(7.2))                   # the usual cut-off applies
print(is_elevated(7.2, 10.0))             # day 2 after surgery: a higher cut-off
print(is_elevated(7.2, threshold=10.0))   # the same, and clearer

### Type hints and docstrings
* A **type hint** records what type each parameter and the return value should be: `name: type`, and `-> type` after the brackets. Python does not enforce them, but editors and readers use them.
* A **docstring** is a string right after the `def` line that says what the function does. It is what `help()` and your editor show.

In [ ]:
def calculate_bmi(height_m: float, weight_kg: float) -> float:
    """Calculate BMI from height in metres and weight in kilograms."""
    bmi = weight_kg / height_m ** 2
    return bmi

print(calculate_bmi(1.80, 90))

# The docstring, as help() would show it
print(calculate_bmi.__doc__)

**» Your Turn** — Fill the blanks to write a function for **mean arterial pressure**: MAP = diastolic + (systolic - diastolic) / 3. Test it with 120/80; the answer should be about 93.3.

In [ ]:
def calculate_map(____, ____):
    """Mean arterial pressure (mmHg) from systolic and diastolic pressure (mmHg)."""
    map_mmhg = ____
    return ____

print(calculate_map(120, 80))   # should print 93.33...

**» Your Turn** — On your own: write `bmi_category(bmi)` that **returns** `'underweight'` below 18.5, `'healthy'` below 25, `'overweight'` below 30, and `'obese'` otherwise (an `if` / `elif` chain, as last session). Then call it with the result of `calculate_bmi(1.80, 90)` and print the category.

In [ ]:
def calculate_bmi(height_m, weight_kg):
    return weight_kg / height_m ** 2

# Your code here

*Finished quickly?* Add type hints and a docstring to `bmi_category`. Then make `calculate_bmi` **check its inputs**: a height of 180 is probably centimetres, and a height of 90 with a weight of 1.8 is the arguments the wrong way round. Decide what ranges are plausible for a height in metres and a weight in kilograms, and what the function should do when a value falls outside them.

## Importing functions and modules
* The functions so far are **built in**: always available.
* Many more live in **modules** of the Python Standard Library, and must be **imported** first. A module is a file of definitions: functions, constants, classes.
* The full list is in the [standard library docs](https://docs.python.org/3/library/index.html), from *Data Types* onwards.

In [ ]:
# Import a whole module...
import math

# ...then access its definitions with module.name
print(math.pi)
print(math.log2(10))

### Importing specific components
* `from X import Y` imports only what you name, so you use `pi` rather than `math.pi`.
* Keeps the namespace tidy, but two modules can define the same name, and the second import silently replaces the first.

In [ ]:
from math import pi, log2

print(pi)
print(log2(6))

# Functions have docstrings too
print(log2.__doc__)

### Module aliases
* `import X as Y` gives a module a short name.
* For the libraries you will use most, the aliases are universal, and you will see them in every tutorial and answer online:
    * `import numpy as np`
    * `import pandas as pd`
    * `import matplotlib.pyplot as plt`

**» Predict** — Four imports, four calls. For each cell decide: **works, or error?** Then run them one at a time.

In [ ]:
# A
import math
print(math.sqrt(16))

In [ ]:
# B
import math
print(sqrt(16))

In [ ]:
# C
from statistics import mean
print(mean([72, 118, 95, 134]))

In [ ]:
# D
import statistics as stats
print(statistics.mean([72, 118, 95, 134]))

### The rule
The name you can use afterwards is **exactly the one written after `import`, or after `as`**, and nothing else.

* `import math` gives you `math`, so it is `math.sqrt`, not `sqrt`.
* `from statistics import mean` gives you `mean`, and not `statistics`.
* `import statistics as stats` gives you `stats`, and not `statistics`. The alias **replaces** the name, which is why every pandas example starts with `import pandas as pd` and then uses `pd`.

## External packages
* Beyond the standard library there are hundreds of thousands of packages on the Python Package Index, [pypi.org](https://pypi.org/).
* Before a package can be imported it must be **installed**: in a Colab code cell `%pip install <package>`, in a terminal `pip install <package>`.
* Colab has today's packages preinstalled, so `import pandas as pd` and `import matplotlib.pyplot as plt` just work. You will install one later in the course.

# 2. CRP results for one patient
Putting Session 1 and today's functions together. **Work in your pair, 10 minutes.**

A ward has recorded 20 C-reactive protein (CRP) results for four patients, in two lists in the same order:

* `patients` holds who each result belongs to, e.g. `'Yusuf'`
* `crp` holds the results as text with a unit, e.g. `'0.1912 mg/L'`

1. Write `count_results(crp, patients, name)` that **returns how many results** the named patient has. Call it for `'Noah'`.
2. Write a second function, `average_crp(crp, patients, name)`, that **returns the mean CRP** for that patient. Start from a copy of the first.

**Suggested approach**

1. Loop over the indices with `range(len(crp))`, as in the prescription chart last session; the two lists line up by position.
2. If `patients[index]` is the name you want, add one to a count that starts at 0 *before* the loop. Return the count.
3. For the mean, also keep a running total: strip the unit from `crp[index]` (`replace` or a slice), cast to `float`, and add it. Return the total divided by the count.

*Finished?* Call it for each of the four names. Which patient has the highest average?

In [ ]:
# CRP results for four patients: two lists in the same order
patients = ['Yusuf', 'Amara', 'Noah', 'Priya', 'Noah', 'Noah', 'Amara', 'Priya', 'Noah', 'Noah',
            'Yusuf', 'Noah', 'Priya', 'Amara', 'Amara', 'Priya', 'Yusuf', 'Yusuf', 'Yusuf', 'Yusuf']
crp = ['0.1912 mg/L', '0.0697 mg/L', '6.063 mg/L', '8.7888 mg/L', '6.1414 mg/L', '5.8497 mg/L', '2.4352 mg/L',
       '2.4059 mg/L', '7.8755 mg/L', '9.4961 mg/L', '0.1357 mg/L', '4.4206 mg/L', '10.7717 mg/L', '0.0268 mg/L',
       '0.0281 mg/L', '4.0544 mg/L', '0.0382 mg/L', '0.0034 mg/L', '0.6772 mg/L', '1.7951 mg/L']

# 1. How many results does the named patient have?
def count_results(crp, patients, name):
    ____

print(count_results(crp, patients, 'Noah'))   # 6

# 2. A second function: the mean CRP for the named patient
def average_crp(crp, patients, name):
    ____

print(average_crp(crp, patients, 'Noah'))     # about 6.64

# 3. Tables in pandas
Hospital data are tables. This section is what a table is in Python, and how to load, inspect, select and filter one. First, your prep work.

## From your feature cards to tables
**Presentations: 2 minutes per group, from your Miro frame (link on Canvas).** In order: A1, A2, B1, B2, C1, C2.

* **One concept with its two routes:** the two tables you could get it from, and which you trust more and why.
* **The recipe for one feature:** read out your green sticky, step by step.

Pre-op and post-op groups on the same operation: listen for what the operation and early recovery added.

## Every recipe had the same verbs
Whatever the feature, the green sticky said the same five things, and they are what the rest of today teaches:

| Your recipe | In Python |
| --- | --- |
| Filter the rows | selecting and filtering a table (this section) |
| Keep the time window | dates and times (appendix today, more in Session 4) |
| Summarise to one value | descriptive statistics and `groupby` (section 4) |
| Join on the patient | `merge` (section 5) |
| Fill the gaps | missing values appear today as `NaN`; handling them is Session 4 |

Hospital data are **tables**: one per system, many rows per patient, each row with a time. The Python library for tables is **pandas**.

## pandas
* Most of the data science and AI methods in this course start from a **table** of data, and `pandas` is the Python library for tables: [pandas.pydata.org](https://pandas.pydata.org/).
* It reads, manipulates, summarises and writes data, is fast on large tables, and plugs into the plotting and machine learning libraries we use later.
* The universal import is `import pandas as pd`. Everything in pandas is then `pd.something`.

### Setup: one import, used by every cell that follows
* A notebook is **one environment**. `pd` imported here, and `DATA` defined here, exist in every cell below, so neither is repeated.
* The same goes for every table we load: load it once, use it in the cells that follow.
* After a runtime restart they are gone again: **Runtime → Run before** brings everything back.

In [ ]:
import pandas as pd

# Where the course data files live, on the course GitHub repository
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

## DataFrames: tabular data
* Tabular (2-dimensional) data are the foundation of most analyses in data science.
* In pandas, and in many other tools, a table is a `DataFrame`.

![pandas data frame](https://pandas.pydata.org/docs/_images/01_table_dataframe.svg)
<small><p>Copyright 2024 <a href="https://pandas.pydata.org/docs">pandas</a> - <a href="https://opensource.org/license/bsd-3-clause">BSD 3-Clause License</a></p></small>

### DataFrame structure
A small example DataFrame, `df`, of five admissions, loaded from a CSV file on the course GitHub repository:

* Six **columns**, each named with a string (`patient_id`, `age`, ...). Each column is a `Series`.
* Five **rows**, with an **index** that starts at zero.
* Columns can have **different types** (whole numbers, decimals, text, a date stored as text), but every value in one column has the same type.
* This one is tiny; a DataFrame can hold gigabytes.

In [ ]:
df = pd.read_csv(DATA + "ward_admissions.csv")
df

### DataFrame properties
* Attributes describe the table: `shape` is (rows, columns), `columns` the names, `dtypes` the type of each column. The full list is in the [DataFrame reference](https://pandas.pydata.org/docs/reference/frame.html).
* These are the first three things to look at after loading any file.

In [ ]:
df.shape               # (number of rows, number of columns)

In [ ]:
df.columns.tolist()    # the column names

In [ ]:
df.dtypes              # the type of each column

### Selecting columns
* One name in square brackets gives a column, a **Series**: `df['ward']`.
* A **list** of names gives a new **DataFrame**: `df[['ward', 'age']]`. Note how differently the two display; some methods exist on a Series only.

In [ ]:
df['ward']              # one column (a Series)

In [ ]:
df[['ward', 'age']]     # several columns (a DataFrame)

### No loops
In *CRP results for one patient* you converted twenty values with a loop: an index, a condition, a cast, one value at a time.

`df['height_m']` holds five heights in metres and you want all five in centimetres. **What would you *like* to be able to write?**

In [ ]:
# Every height in centimetres
heights_cm = ____
heights_cm

### A column is not a list
* A Series applies the operation to **every value at once**: five heights in, five heights in centimetres out. No loop, no index.
* This is the whole shift from last session. The twenty-value loop in *CRP results for one patient* becomes one line, and the same line works on five rows or five million.
* That task also needed two lists kept in step by index. A DataFrame keeps each **row together**, so a filter on one column brings every other column with it.
* Comparison and logical operators work the same way, one result per row.

### Creating new columns
* Assign to a column name that does not exist yet, and it is created.
* Here BMI for every patient in one line: the CRP loop, without the loop.

In [ ]:
df['bmi'] = (df['weight_kg'] / df['height_m'] ** 2).round(1)   # arithmetic on two columns, row by row
df['overweight'] = df['bmi'] >= 25                               # a comparison, row by row
df

# 4. Independent practice
From here to **Combining tables** is yours, at your own pace, about an hour: the rest of the table basics (rows, filtering, modifying) and the CRP task, then **Reading and writing files**, **Cleaning text columns**, **Descriptive statistics** and **Grouping**.

For each idea:

1. **Read** the short explanation.
2. **Run** the worked example yourself.
3. Do the **» Predict** and **» Your Turn** tasks.

Run the cells in order: each builds on the ones before it. Work together in your pair, and call me over when you get stuck.

**Understanding beats finishing.**

### Selecting rows and single values
* `.loc[...]` selects by **label**: a row label on its own, or a row label and a column name together.
* It takes **square brackets**, like a list, not round ones: `df.loc[1]`, not `df.loc(1)`.
* Lists of labels select several rows or columns at once.
* `.iloc[...]` is the same by **position**, counting from zero.

In [ ]:
df.loc[1]                          # one row, by its label

In [ ]:
df.loc[1, 'ward']                  # one value: row label and column name

In [ ]:
df.loc[[0, 2], ['ward', 'age']]    # several rows and several columns

**» Predict** — For `df` as loaded above, **write down what each cell returns**, then run them one at a time. For the last, how many rows?

In [ ]:
df['age']

In [ ]:
df.loc[2]

In [ ]:
df.loc[4, 'ward']

In [ ]:
df.loc[1:3]

### Two methods you will use constantly
* `head()` shows the first rows. With five rows you can see the whole table; with thousands or millions you cannot, and `head()` is how you look at data at all, and the quickest check that a step did what you expected.
* `describe()` summarises the numeric columns.
* Others to look up when you need them: `info()`, `sort_values()`, and the rest of the [DataFrame methods](https://pandas.pydata.org/docs/reference/frame.html).

In [ ]:
df.head(3)         # the first rows (default 5): the quickest check that a step did what you expected

In [ ]:
df.describe()      # summary statistics of the numeric columns

### Filtering rows
* Square brackets with a **list of booleans**, one per row, keep the `True` rows.
* Nobody types that list: a comparison on a column **produces** it. So any condition on the columns filters the table.
* "All patients treated by one surgeon", "all results above a threshold": this is the pattern.

In [ ]:
df[[True, False, False, False, True]]    # keep rows 0 and 4

**» Predict** — Look at the table. **How many rows does this return?**

In [ ]:
df[df['ward'] == 'Surgical']

In [ ]:
print(df['bmi'] >= 25)     # the boolean Series a comparison produces...
df[df['bmi'] >= 25]        # ...used as the filter

**» Predict** — No patient is over 100. **What do you think this returns: an error, nothing, or something else?** Then run it, and think about what that means for a filter with a typo in its threshold.

In [ ]:
df[df['age'] > 100]

### Modifying values
* The same accessors on the **left** of `=` change the table: a whole column, a row, or one value.

In [ ]:
df['ward'] = ['Surgical', 'Surgical', 'Surgical', 'Medical', 'Medical']   # replace a column
df.loc[4] = [1006, 51, 77, 1.69, 'ICU', '2026-01-08', 27.0, True]         # replace a row: one value per column
df.loc[1, 'ward'] = 'ICU'                                                 # replace a single value

df

**» Your Turn** — Back to the CRP results, now as a table. Fill the blanks to load the file, look at it, and keep only **elevated** results above 5.0 mg/L. The `.str` line removes the unit from every value at once; you will meet it properly in the next block.

In [ ]:
# 1. Load the table and look at the first rows
df_crp = pd.read_csv(DATA + "crp_results.csv")
print(df_crp.____())

# 2. A numeric copy of the crp column (removes the unit, then casts to float)
df_crp['crp_value'] = df_crp['crp'].str.removesuffix(' mg/L').astype(float)

# 3. Keep only the rows where crp_value is above 5.0
elevated = df_crp[____]

# 4. How many rows are left? Then look at them
print(____)
elevated

## Reading and writing files
* pandas reads many formats into a DataFrame: `pd.read_csv`, `pd.read_excel`, `pd.read_sql`, `pd.read_json`, ...
* ...and writes a DataFrame back out with the matching **method**: `df.to_csv`, `df.to_excel`, `df.to_sql`, `df.to_json`, ...
* Both need a **filepath**: where the file is. On your own computer that is something like `C:\Users\You\Documents\MyData.xlsx`; in Colab there is no local disk, so the course data are read from **web URLs**. Uploading your own files, or using Google Drive, is in the appendix *Loading your own data*.

### CSV files
* Comma Separated Values: a text file, one row per line, values separated by commas, usually with the column names on the first line.

```
patient_id, date_of_birth, height, weight
001, 2001-04-13, 1.65, 67
002, 1982-11-23, 1.45, 52
```

* `pd.read_csv` takes the first line as the column names, makes a good guess at each column's type (numbers become numbers; dates usually stay text), and has options for every odd format you will meet.

In [ ]:
df_crp = pd.read_csv(DATA + 'crp_results.csv')
df_crp.head()

### Writing a CSV
* `to_csv()` on a DataFrame writes it to a path. In Colab the file appears in the **Files** panel (folder icon, left), from where you can download it.
* `index=False` leaves out the row numbers, which you rarely want in a file.
* The `read_` functions and `to_` methods are all highly configurable: see [the IO docs](https://pandas.pydata.org/docs/user_guide/io.html).

In [ ]:
df_crp.to_csv('crp_copy.csv', index=False)

**» Your Turn** — Load `str_data.csv` (ward observations for 2000 patients). Print its shape and its column types, and look at the first rows. **Which columns are numbers, and which are text that merely looks like numbers?**

In [ ]:
# Your code here

## Cleaning text columns
* Real data arrives with the unit, a label, or two values in one cell: `54.3 bpm`, `UNRESPONSIVE_3`, `96/72 mmHg`.
* pandas reads these as **text** (shown as `str` or `object` by `dtypes`), so no arithmetic or statistics are possible until they are cleaned.
* The `.str` attribute of a text column applies a string method to **every value at once**. Last session's `split`, `replace`, `upper` are all there: [method summary](https://pandas.pydata.org/docs/user_guide/text.html#method-summary).

In [ ]:
df = pd.read_csv(DATA + "str_data.csv")
df.head()

### Example 1: removing a unit
* `removesuffix()` takes a fixed ending off every value (there is also `removeprefix()`).
* Here it removes `bpm` from the heart rates.

In [ ]:
df['hr_clean'] = df['hr'].str.removesuffix('bpm')
df.head()

**» Predict** — `hr_clean` looks like numbers. **Does `.mean()` work on it?** Decide, then run.

In [ ]:
hr_clean = df['hr'].str.removesuffix('bpm')
print(hr_clean.mean())

### Still text
* `removesuffix` returns strings, so the column still cannot be averaged. **Cast** it with `.astype(float)`, exactly as `float()` did for a single value last session.
* Watch for whitespace left behind: `'54.3 '` casts fine, but a stray letter does not.

In [ ]:
df['hr_numeric'] = df['hr'].str.removesuffix('bpm').astype(float)
df.head()

In [ ]:
df['hr_numeric'].mean()

### Example 2: splitting a value
* `conscious_level` is a label and a score joined by `_`, so there is no fixed prefix to remove.
* `split('_')` gives a list for each value; `.str[1]` takes the second item from each list; then cast.
* Chaining methods like this is how most pandas cleaning reads: left to right, one step per method.

In [ ]:
df['conscious_level_score'] = df['conscious_level'].str.split('_').str[1].astype(int)
df.head()

**» Your Turn** — Fill the blanks to pull **systolic** and **diastolic** pressure out of `bp` (`96/72 mmHg`) as integers, then compute **pulse pressure** (systolic minus diastolic).

In [ ]:
# Take the unit off first, so both numbers are clean
bp = df['bp'].str.removesuffix(' mmHg')

df['systolic'] = bp.str.split('/').str[____].astype(____)
df['diastolic'] = bp.str.split('/').str[____].astype(____)
df['pulse_pressure'] = ____
df.head()

## Descriptive statistics
* A numeric Series has the statistics built in: `mean()`, `median()`, `std()`, `min()`, `max()`, `count()`, and more in the [docs](https://pandas.pydata.org/docs/reference/frame.html#computations-descriptive-stats).
* They only make sense for numbers, which is the reason to clean text columns first.

In [ ]:
print('HR mean:  ', df['hr_numeric'].mean())
print('HR st dev:', df['hr_numeric'].std())
print('HR min:   ', df['hr_numeric'].min())
print('HR max:   ', df['hr_numeric'].max())

**» Your Turn** — On your own: print the **mean, median and standard deviation** of your `pulse_pressure` column from the previous task. Is the median close to the mean?

In [ ]:
# Your code here

## Grouping: statistics per group
* A statistic over the whole table is rarely the question. You want it **per patient, per surgeon, per ward**.
* `groupby` takes one or more columns and splits the table into groups; any statistic that follows is computed **group by group**.
* Here is the *CRP results for one patient* task, in three lines:

In [ ]:
df_crp['crp_value'] = df_crp['crp'].str.removesuffix(' mg/L').astype(float)

df_crp.groupby('patients')['crp_value'].mean()

Read the last line right to left: the mean, of `crp_value`, within each group of `patients`. `groupby(...)` on its own returns a grouped object; nothing is computed until you ask for a statistic.

**» Your Turn** — Fill the blanks: flag elevated results, then **count how many each patient has**. `True` counts as 1, so the sum of a boolean column is a count.

In [ ]:
df_crp['elevated'] = df_crp['crp_value'] ____ 5.0
df_crp.groupby(____)['elevated'].____()

## Regroup

We will go through **Combining tables** and **Plotting** together.

If you didn't finish the block above, you can finish it after the session.

Finished early? Help a neighbour, or start the **Additional work**.

# 5. Combining tables
Clinical data arrive in several tables, so combining them is routine. Two cases:

* **Stacking**: tables with the **same columns**, e.g. this year's and last year's admissions. One on top of the other: `pd.concat`.
* **Linking**: tables with **different columns about the same entities**, e.g. demographics in one and outcomes in another. Side by side, matched on a key: `pd.merge`.

### Concatenation
* `pd.concat` takes a **list** of DataFrames and stacks them, so any number can be combined in one call.
* The columns should match; the row index is kept from each input, so expect repeated labels.

<img src="https://pandas.pydata.org/docs/_images/merging_concat_basic.png" alt="pandas concat tables" width="620"/>
<small><p>Copyright 2024 <a href="https://pandas.pydata.org/docs">pandas</a> - <a href="https://opensource.org/license/bsd-3-clause">BSD 3-Clause License</a></p></small>

### Concatenation example
Adult and paediatric demographics, recorded separately, stacked into one table.

In [ ]:
df_adult = pd.read_csv(DATA + "demographics_adult.csv")
df_paediatric = pd.read_csv(DATA + "demographics_paediatric.csv")

df_concat = pd.concat([df_adult, df_paediatric])

print("Rows in adult: ", df_adult.shape[0])
print("Rows in paeds: ", df_paediatric.shape[0])
print("Rows in concat:", df_concat.shape[0])

### Stacking the wrong tables
Recovery time after surgery is recorded in a **different table**, with different columns: a patient `id` and `recovery_days`.

In [ ]:
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
df_recovery.head(2)

**» Predict** — Two rows of demographics stacked on two rows of recovery times, columns that do not match. **Error, or a table? If a table, what fills the gaps?**

In [ ]:
pd.concat([df_adult.head(2), df_recovery.head(2)])

No error: pandas lines columns up **by name** and fills the gaps with `NaN` (not a number: missing). Stacking the wrong tables therefore fails silently, so look at the result after every concat.

### Merging
* `pd.merge` links two tables through a **key** that exists in both, identifying which rows describe the same entity: a patient, an operation, a clinician, a device.
* Keys are numbers or short codes (`012345`), never names, to avoid confusion.

<img src="https://pandas.pydata.org/docs/_images/merging_merge_on_key.png" alt="pandas merge tables" width="620"/>
<small><p>Copyright 2024 <a href="https://pandas.pydata.org/docs">pandas</a> - <a href="https://opensource.org/license/bsd-3-clause">BSD 3-Clause License</a></p></small>

### Merging example
* Does recovery time after surgery differ by gender? Gender is in the **demographics** table, recovery time in the **recovery** table, and both have the patient `id`.

In [ ]:
df_demographics = pd.read_csv(DATA + "demographics.csv")
df_demographics.head()

In [ ]:
df_recovery.head()

* `pd.merge` calls its inputs **left** and **right**; `on` names the key column (several are allowed).

In [ ]:
df_merged = pd.merge(df_demographics, df_recovery, on='id')
print(df_merged.shape)
df_merged.head()

**» Predict** — `recovery_sample` is the first 10 rows of the recovery table. `df_demographics` has 1000 rows. **How many rows does the merge have?** Then run.

In [ ]:
recovery_sample = df_recovery.head(10)

df_sample_merge = pd.merge(df_demographics, recovery_sample, on='id')
print(df_sample_merge.shape)

### Which rows are kept: the merge type
* By default `merge` is an **inner** join: only keys present in **both** tables survive, so 990 patients vanished without a message.
* The `how` parameter changes this ([docs](https://pandas.pydata.org/docs/user_guide/merging.html#merge-types)):
    * `inner`: keys in both (the default)
    * `left`: every row of the left table, with `NaN` where the right has no match
    * `right`: every row of the right table
    * `outer`: every key from either table
* Repeated keys multiply rows: a patient with three lab results merged with demographics gives three rows. Always check `.shape` before and after.
* *Additional work 2* tries the other merge types on the sample above.

# 6. Plotting
* What do you do when a few summary numbers no longer describe your data?
* How do you understand the **distribution** of a measurement across patients?
* You suspect older patients recover more slowly. What is more convincing than a correlation coefficient?

## matplotlib
* `matplotlib` is the mature plotting library for Python: [matplotlib.org](https://matplotlib.org/). Line charts, bar charts, histograms, scatter plots, box plots, and anything else you can describe.
* Plots are built up in **stages**, then drawn with `plt.show()`. If you have used MATLAB this will feel familiar.
* The universal import is `import matplotlib.pyplot as plt`, once, as with pandas.

In [ ]:
import matplotlib.pyplot as plt

### A basic plot
* `plt.plot(values)` draws a line through the values, against their position (0, 1, 2, ...). Here the position is the post-operative day.
* Labels and a title are added by further calls, then `plt.show()` draws the figure.

In [ ]:
temperature = [37.8, 38.4, 38.1, 37.6, 37.2, 36.9]   # one reading per post-op day
plt.plot(temperature)
plt.title('Post-operative temperature')
plt.ylabel('Temperature (°C)')
plt.show()

### Two lines on one figure
* Everything you add goes on the **current figure** until `plt.show()` draws it and starts afresh. So two `plot()` calls and one `show()` give one figure with two lines.
* Build a plot up in stages and show it once; for two separate figures, call `show()` twice.

In [ ]:
systolic = [128, 132, 118, 110, 124, 126]
diastolic = [82, 85, 76, 70, 78, 80]
plt.plot(systolic)
plt.plot(diastolic)
plt.title('Blood pressure')
plt.show()

### A 2D plot with styling
* Give `plot()` the x values as well, and it plots points. Here the readings are not evenly spaced in time, so the position is not the right x.
* `color`, `linestyle` and `marker` control the look of each series.

In [ ]:
hours = [0, 6, 12, 24, 48, 72]            # hours after surgery
crp = [5, 40, 95, 150, 120, 60]           # mg/L
plt.plot(hours, crp, color='red', linestyle='--', marker='o')
plt.xlabel('Hours after surgery')
plt.ylabel('CRP (mg/L)')
plt.show()

**» Your Turn** — Two patients' CRP after the same operation. Fill the blanks to plot both on one figure, each with a `label`, and add a legend with `plt.legend()`. Which patient would you be worried about?

In [ ]:
hours = [0, 6, 12, 24, 48, 72]
crp_a = [5, 40, 95, 150, 120, 60]
crp_b = [8, 60, 140, 210, 250, 230]

plt.plot(hours, crp_a, label=____, color='blue', marker='o')
plt.plot(____, ____, label='Patient B', color='red', marker='x')
plt.xlabel(____)
plt.ylabel('CRP (mg/L)')
plt.legend()
plt.show()

## Plotting from pandas
* A Series and a DataFrame both have a `plot()` method that calls matplotlib for you, so most everyday plots are one line.
* The heart-rate file has one observation every four hours, with the time in `obs_time`.

**» Predict** — Plotting the `hr` column on its own. **What do you expect on the x-axis?** Then run.

In [ ]:
df_hr = pd.read_csv(DATA + 'hr_timeseries.csv')

df_hr['hr'].plot()
plt.show()

The **row index**, 0 to 23, not the time. A Series knows its values and its index and nothing else. Name the x column explicitly. A time axis also needs real timestamps rather than text, so parse the column first with `pd.to_datetime` (more in *Additional work 7*):

In [ ]:
df_hr['obs_time'] = pd.to_datetime(df_hr['obs_time'])

df_hr.plot('obs_time', 'hr')
plt.show()

### The plots you need
Every one of these is one line on a DataFrame or a Series. The rest, titles, labels, colours and sizes, is in the [`DataFrame.plot` docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html).

| Question | Plot | One line |
| --- | --- | --- |
| How does y change along x? | line | `df.plot('x', 'y')` |
| Are x and y related? | scatter | `df.plot('x', 'y', kind='scatter')` |
| How do the groups compare? | bar | `df.groupby('group')['y'].mean().plot(kind='bar')` |
| How is y distributed in each group? | box | `df.boxplot(column='y', by='group')` |
| How is y distributed overall? | histogram | `df.hist('y', bins=10)` |

Then `plt.show()`. The capstone, after the Closing, uses all five.

# 7. Closing

## Plots are opinionated
Every plot makes a **claim** about the data and **hides** something. The same forty heart-rate readings at each of four times of day, plotted three ways.

**» Predict** — From the pie chart: **which time of day has the highest heart rate, and by roughly how much?**

In [ ]:
import numpy as np

# Resting heart rate, 40 readings at each time of day (synthetic)
rng = np.random.default_rng(4)
readings = {}
for time_of_day, mean in [('night', 74), ('morning', 76), ('afternoon', 78), ('evening', 75)]:
    x = rng.normal(mean, 9, 40)
    readings[time_of_day] = x - x.mean() + mean
means = pd.Series({k: v.mean() for k, v in readings.items()})

means.plot.pie(title='Mean heart rate by time of day')
plt.ylabel('')
plt.show()

The same four means as a bar chart. **Which is highest now, and by how much?**

In [ ]:
means.plot.bar(rot=0, ylim=(70, 80), title='Mean heart rate by time of day')
plt.ylabel('Heart rate (bpm)')
plt.show()

And the readings themselves.

In [ ]:
plt.boxplot(list(readings.values()), tick_labels=list(readings.keys()))
plt.ylim(40, 110)
plt.title('Heart rate by time of day: the readings')
plt.ylabel('Heart rate (bpm)')
plt.show()

### What each plot claimed
* **The pie** claims the four means are parts of one whole, which is false, and asks you to compare angles, which nobody can. It says "all the same". Pie charts are never the right plot.
* **The bar** claims "compare these lengths", and with the axis starting at 70 a 4 bpm gap looks like a doubling. Bars start at zero, or you say plainly that you zoomed in.
* **The box plots** show the data: the differences are real but a few bpm against a spread of nine, and the groups overlap almost entirely. A summary hides what the readings show.

### Before you hand in a figure
1. **The title states the finding**, not just the variable name.
2. **Every axis is labelled, with units.**
3. **The axis range is honest**: bars start at zero, or you say that you zoomed in.
4. **Groups are distinguishable**: by position, colour, and a legend or label.
5. **One message per plot**: if you need two, make two plots.

The capstone's figures are the first to apply it to.

## Wrap up
* This session does not cover every method you will need for later tasks or your project. Use the documentation and search: the question you have has been asked before.
* There are usually several ways to get the same result. For now, prioritise **readability > performance > cleverness**.
* pandas and matplotlib are not the only options. `seaborn` sits on top of matplotlib with a higher-level interface for statistical plots; `plotly` makes interactive figures for the web. Both take pandas DataFrames.

## Before Session 3
1. **Do the capstone** below, *Does recovery depend on who the patient is?*, and any » Your Turn tasks you did not reach. Check your answers against the solutions notebook once it is released. Then try the **Additional work** below: *Lab tests and recovery* extends the capstone and is the most valuable.
2. **MIMIC-IV access must be in place for Session 3**, which queries it. If your credentialling has not come through, chase it now: instructions on [Canvas](https://canvas.imperial.ac.uk/courses/8306/modules/items/1684250).
3. **Preparation for Session 3:** see [Canvas](https://canvas.imperial.ac.uk/courses/8306).
4. The reading list has tutorials that extend today's material, resources on data visualisation, and an introduction to `numpy`, the numerical library underneath pandas.

# 8. Capstone: does recovery depend on who the patient is?

This ties together merge, groupby and every plot type, and it is the main piece of work **before Session 3**: started in the session if there is time, finished at home. There is no worked example for this one. It is one of your feature cards, end to end: join two tables on the patient, summarise per group, and look at the result.

Two tables from one surgical service: **demographics**, one row per patient (`id`, names, `date_of_birth`, `gender`, `ethnicity_code`), and **recovery** (`id`, `recovery_days` after surgery). The service wants to know whether recovery time differs between groups of patients.

**Tasks** — go as far as you can:

1. **Merge** the two tables on `id`. Check the shape: you should still have 1000 rows.
2. **Histogram** of recovery days. Try `bins=10` and `bins=40`: which tells the truth better?
3. **Mean recovery time by gender**, with `groupby`, then the same means as a **bar chart**.
4. **Box plot** of recovery days by gender. What does it show that the bar chart hides?
5. **Age:** make a `birth_year` column by slicing the first four characters of `date_of_birth` with `.str[:4]` and casting to `int`. Then a **scatter plot** of birth year against recovery days. Is there a relationship?
6. *Stretch:* there are 17 ethnicity codes, each with few patients. They are the [NHS Data Dictionary ethnic category codes](https://archive.datadictionary.nhs.uk/DD%20Release%20July%202026/data_elements/ethnic_category.html), which the dictionary groups as `'ABC'` White, `'DEFG'` Mixed, `'HJKL'` Asian or Asian British, `'MNP'` Black or Black British, `'RS'` Other Ethnic Groups, and `'Z'` Not stated. Write a function that maps a code to its group, apply it to the column with `df['ethnicity_code'].apply(your_function)`, and box plot recovery by group.

**Suggested approach:**

1. The merge is the worked example from Combining tables; print `.shape` straight after it.
2. to 5. The one-liners are on *The plots you need*; `groupby('gender')['recovery_days'].mean()` is a Series, so `.plot(kind='bar')` goes straight on the end.
6. `'A' in 'ABC'` is `True`: an `if` / `elif` chain on that does the mapping. `.apply()` calls your function on every value of the column and gives back the results as a new column.

> A worked solution is in the solutions notebook, released after the session.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")

# Your code here

# Additional work
More practice for after the session. Worked solutions are in the solutions notebook, released after the session.

## Additional work 1 — Lab tests and recovery (extends the capstone)

`laboratory_tests.csv` is a **long** table: one row per test (`id`, `patient_id`, `test`, `value`), three tests per patient (`crp`, `wbc_z_score`, `albumin_level`). The values are text, and the CRP results mix units (`mg/L` and `g/L`).

1. **Merge** it with the recovery table. The key has different names in the two tables: `left_on='patient_id', right_on='id'`. Check the shape: three rows per patient.
2. **Pivot** to one row per patient with a column per test (see *Additional work 4*): `pivot(index=['patient_id', 'recovery_days'], columns='test', values='value')`, then `.reset_index()`.
3. **WBC:** cast `wbc_z_score` to float and scatter it against recovery days.
4. **CRP:** split each value into number and unit (`.str.split(' ', expand=True)`), cast the number, multiply the `g/L` rows by 1000, then scatter against recovery days.
5. **Albumin:** the labels differ only in case. Lower-case them, then box plot recovery by albumin level.
6. *Stretch:* bin CRP with `pd.cut`, and box plot recovery by `['gender', 'crp_bins']` after merging gender back in. Is the CRP association explained by gender?

In [ ]:
df_lab = pd.read_csv(DATA + "laboratory_tests.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
df_demographics = pd.read_csv(DATA + "demographics.csv")

# Your code here

## Additional work 2 — Merge types

**» Predict** — The sample merge from the session, with `how='left'`. The last line counts patients with no recovery time: `isna()` marks missing values as `True`, and the sum of booleans is a count. **Write down the shape and the count**, then run.

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
recovery_sample = df_recovery.head(10)

df_left = pd.merge(df_demographics, recovery_sample, on='id', how='left')
print(df_left.shape)
print(df_left['recovery_days'].isna().sum())   # patients without a recovery time

**» Your Turn** — Now `how='right'` and `how='outer'`: predict the shape of each before running. Then merge `df_demographics` with the **whole** recovery table using each of the four types. Why are all four the same shape?

In [ ]:
import pandas as pd
DATA = "https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/data/"

df_demographics = pd.read_csv(DATA + "demographics.csv")
df_recovery = pd.read_csv(DATA + "simple_recovery.csv")
recovery_sample = df_recovery.head(10)

# Your code here

## Additional work 3 — Subplots and saving figures
* `plt.subplot(nrows, ncols, index)` places the next plot in a grid: useful for synchronised physiological signals on different scales, or the axes of a robot's movement.
* `plt.tight_layout()` stops the panels overlapping.

In [ ]:
hours = [0, 6, 12, 24, 48, 72]
crp_a = [5, 40, 95, 150, 120, 60]
crp_b = [8, 60, 140, 210, 250, 230]

plt.subplot(1, 2, 1)
plt.plot(hours, crp_a)
plt.title('Patient A')

plt.subplot(1, 2, 2)
plt.plot(hours, crp_b)
plt.title('Patient B')

plt.tight_layout()
plt.show()

* `plt.savefig(path)` writes the current figure to an image file, **before** `plt.show()` clears it. In Colab the file appears in the Files panel.

In [ ]:
temperature = [37.8, 38.4, 38.1, 37.6, 37.2, 36.9]
plt.plot(temperature)
plt.savefig('temperature.png')
plt.show()

**» Your Turn** — Plot the heart-rate time series and a histogram of the same values side by side, then save the figure as `hr.png`.

In [ ]:
df_hr = pd.read_csv(DATA + 'hr_timeseries.csv')

# Your code here

## Additional work 4 — Reshaping: long and wide tables
* **Long**: one row per **observation**, so several rows per subject. Produced by most clinical systems; easy to combine and group.
* **Wide**: one row per **subject**, one column per observation. Easier for a human to read, and what regression and machine learning libraries expect.
* `pivot` goes long to wide; `melt` goes wide to long. A subject may be a patient, an admission, an instrument: anything observed more than once.

### Pivot (long to wide)
* Monitoring data from five patients, one row per sensor reading.
* `pivot` needs the column for the rows (`index`), the column whose values become the new columns (`columns`), and the column to fill the cells (`values`). The `taken_datetime` column is dropped.

In [ ]:
df_long = pd.read_csv(DATA + "long_monitoring.csv")
df_long.head(8)

In [ ]:
df_wide = df_long.pivot(
    index='patient_id',     # one row per patient
    columns='sensor',       # one column per sensor
    values='value'          # the cell contents
)
df_wide

**» Predict** — `df_long_corrupt` is the same table with the first two rows **duplicated**, so one patient has two readings from the same sensor. **Does `pivot` keep the first, keep the last, or refuse?** Then run.

In [ ]:
df_long_corrupt = pd.concat([df_long.head(2), df_long])

df_long_corrupt.pivot(index='patient_id', columns='sensor', values='value')

* `pivot` refuses: it cannot know which reading you want. `pivot_table` is the flexible version, with `aggfunc` to combine repeats and `fill_value` for missing cells.

In [ ]:
df_long_corrupt.pivot_table(
    index='patient_id',
    columns='sensor',
    values='value',
    aggfunc='min',              # how to combine repeated observations
    fill_value='MISSING'        # what to put in empty cells (default: NaN)
)

### Melt (wide to long)
* Robot state, one row per instrument with its position and state in columns.
* `melt` names the identifier columns (`id_vars`) and the columns to unpivot (`value_vars`).

In [ ]:
df_wide = pd.read_csv(DATA + "wide_robot.csv")
df_wide

In [ ]:
df_wide.melt(
    id_vars=['instrument_id'],
    value_vars=['x_pos', 'y_pos', 'z_pos', 'state']
)

### Quick check
1. Which operation combines two tables that have the same columns?
2. What is a *wide* table?
3. Which operation turns a long table into a wide one?
4. Which operation combines two tables that hold different information about the same patients?

## Additional work 5 — Regular expressions
* A regular expression (**regex**) describes a **pattern** in text. Powerful, widely used for extracting structure from text, and famously hard to read.
* Several pandas string methods take a regex: `str.contains()`, `str.extract()`, `str.replace()`, `str.match()`.
* This is a taste; the reading list has a proper tutorial.

### Literals
* A literal matches itself, exactly. `'POSITIVE'` matches `POSITIVE` and nothing else, not `positive`.

In [ ]:
df = pd.read_csv(DATA + "str_data.csv", usecols=['microbio_culture'])

df['is_positive'] = df['microbio_culture'].str.contains('POSITIVE')
df['is_positive'].sum()

### Metacharacters and quantifiers
* **Metacharacters** stand for something other than themselves: `.` any character, `^` start of the text, `$` end of the text ([more](https://docs.python.org/3/howto/regex.html#matching-characters)).
* **Quantifiers** say how many times: `*` zero or more, `+` one or more, `{n}` exactly n.
* `POS.*` matches `POS` followed by anything, so `POS` and `POSITIVE`.

In [ ]:
df['is_positive'] = df['microbio_culture'].str.contains('POS.*')
df['is_positive'].sum()

**» Predict** — Some results are recorded as a bare `+`. **Does `str.contains('+')` count them, count nothing, or give an error?** Then run.

In [ ]:
df['microbio_culture'].str.contains('+').sum()

* An error: `+` is a quantifier, and there is nothing in front of it to repeat. To match a metacharacter literally, **escape** it with a backslash, `\+`, written as a raw string `r'\+'` so Python leaves the backslash alone. Or pass `regex=False` to search for plain text.

### Alternation and case
* `|` means **or**: `POS|\+` matches `POS` or a plus sign.
* `case=False` ignores case, so `positive` and `POSITIVE` both match. (Inside the pattern itself, `(?i)` at the start does the same.)
* After a cleaning step like this, **look at what matched**: `value_counts()` on the matched rows.

In [ ]:
df['is_positive'] = df['microbio_culture'].str.contains(r'POS|\+', case=False)

print(df['is_positive'].sum())
print(df[df['is_positive']]['microbio_culture'].value_counts())

### Character classes and special sequences
* A **character class** in square brackets matches any **one** of the characters inside: `[aeiou]` any vowel, `[0-9]` any digit. Brackets inside a class are just characters, so `[(POS)(+)]` does not mean "POS or +".
* **Special sequences** are a backslash and a letter: `\d` a digit, `\w` a word character, `\s` whitespace ([full list](https://docs.python.org/3/library/re.html#regular-expression-syntax)).
* `str.extract()` pulls out the part in **round brackets** (a group). Here it takes the score from the conscious level without knowing the label.

In [ ]:
df = pd.read_csv(DATA + "str_data.csv", usecols=['conscious_level'])

df['score'] = df['conscious_level'].str.extract(r'(\d)').astype(int)
df.head()

## Additional work 6 — Chaining pipelines
* You have been chaining methods all session: `df['bp'].str.removesuffix(' mmHg').str.split('/')`. Each method's output feeds the next.
* Longer chains avoid a clutter of intermediate variables you never use again. A backslash at the end of a line continues the chain on the next.
* For example, the maximum x, y and z position across all instruments in the robot table:

In [ ]:
pd.read_csv(DATA + "wide_robot.csv") \
    .melt(id_vars=['instrument_id'], value_vars=['x_pos', 'y_pos', 'z_pos']) \
    .groupby('variable') \
    ['value'] \
    .max()

## Additional work 7 — Dates and times
* Time series are data with a date and/or time attached to each entry, sampled regularly or not. Clinical observations and robot state are both time series.
* `pd.read_csv` leaves a date-time column as **text**, so no arithmetic is possible: how long between two readings?

In [ ]:
ts = pd.read_csv(DATA + 'hr_timeseries.csv')

print(type(ts.loc[0, 'obs_time']))
ts.head()

### Parsing to datetime
* `pd.to_datetime` parses the text into real timestamps, which can be compared, subtracted and summarised.
* The difference between two timestamps is a `Timedelta`.

In [ ]:
ts['obs_time'] = pd.to_datetime(ts['obs_time'])

print(type(ts.loc[0, 'obs_time']))
print('Gap between readings:', ts.loc[1, 'obs_time'] - ts.loc[0, 'obs_time'])
print('Two hours later:     ', ts.loc[1, 'obs_time'] + pd.Timedelta(hours=2))
print('Is reading 0 earlier?', ts.loc[0, 'obs_time'] < ts.loc[1, 'obs_time'])
print('First reading:       ', ts['obs_time'].min())
print('Last reading:        ', ts['obs_time'].max())

### The dt attribute
* As `.str` gives string methods for a text column, `.dt` gives the parts of a datetime column: `.dt.hour`, `.dt.dayofweek`, `.dt.date`, ... ([components](https://pandas.pydata.org/docs/user_guide/timeseries.html#time-date-components)).
* Combined with `groupby`, this answers questions like "how does heart rate vary by hour of day?"

In [ ]:
ts.groupby(ts['obs_time'].dt.hour)['hr'].describe()

## Additional work 8 — Loading your own data into Colab
Colab has no access to your computer's files, so there are three ways to read your own data:

**1. A web URL**, as the course data: only for data that may be public.

**2. Upload at runtime** with the `files` module of the `google.colab` package, which pops up a file picker. The file lives only as long as the runtime.

```python
from google.colab import files
uploaded = files.upload()          # pick a file; it is then readable by name
df = pd.read_csv('my_data.csv')
```

**3. Google Drive**: mount your Drive, then read files by path. Colab asks for permission.

```python
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/My Drive/path/to/my_data.csv')
```

More in the [Colab IO notebook](https://colab.research.google.com/notebooks/io.ipynb). **Never put data you are not permitted to store overseas into Colab or Drive**: that includes almost all clinical data.